# Train Machine Learning Model on SNOTEL, PRISM, and DEM Data

This notebook will utilize data downloaded and prepared in previous notebooks in training a machine learning model. The model goals are as follows:

__Target variable:__
* SWE (Snow-Water Equivalent) at daily resolution

__Predictor variables:__
* Minimum Temperature
* Maximum Temperature
* Daily Precipitation
* Cumulative Precipitation
* Rolling 3-day mean of Tmin
* Rolling 3-day mean of Tmax
* Elevation
* Slope
* Aspect Northness
* Aspect Eastness

SWE will be predicted using RandomForests or XGBoost Algorithms (talk about why)

## Step 1: Import Libraries and Set Paths

In [18]:
# import libraries

# file management
import os
import pathlib
from pathlib import Path

# datatypes
import numpy as np
import pandas as pd
import xarray as xr

# geospatial data
import geopandas as gpd
import rioxarray as rxr

# machine learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
#import xgboost as xgb

# plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas

In [4]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')

## Step 2: Import Data

In [5]:
# set paths

ml_dir = Path(cleaned_data_dir, 'machine_learning')
train_path = Path(ml_dir, 'stations_train.parquet')
test_path = Path(ml_dir, 'stations_test.parquet')

In [7]:
# read files

if os.path.exists(train_path):
    # import
    stations_train = pd.read_parquet(train_path)
    # print success
    print("Training data successfully imported!")
else:
    print(f"Training data not found. Check {ml_dir} for file.")

if os.path.exists(test_path):
    stations_test = pd.read_parquet(test_path)
    print("Testing data successfully imported!")
else:
    print(f"Test data not found. Check {ml_dir} for file.")

Training data successfully imported!
Testing data successfully imported!


In [8]:
# check out training to remember shape
stations_train

,date,stationTriplet,latitude,longitude,swe_mm,prism_ppt_mm,prism_tmin_c,prism_tmax_c,elevation,slope,aspect_north,aspect_east,prism_tmin_c_roll_3d,prism_tmax_c_roll_3d,prism_ppt_mm_roll_3d,wy_cumul_precip_mm,water_year
0,1997-10-01,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,3.541,15.947000,2996,21.693270,-0.455876,0.890043,3.541000,15.947000,0.000,0.000000,1998
1,1997-10-02,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,6.940,16.136000,2996,21.693270,-0.455876,0.890043,5.240500,16.041500,0.000,0.000000,1998
2,1997-10-03,318:MT:SNTL,44.47147,-112.98191,0.0,5.717,-2.346,9.064000,2996,21.693270,-0.455876,0.890043,2.711667,13.715666,5.717,5.717000,1998
3,1997-10-04,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,-2.784,8.394000,2996,21.693270,-0.455876,0.890043,0.603333,11.198000,5.717,5.717000,1998
4,1997-10-05,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,-2.662,8.978000,2996,21.693270,-0.455876,0.890043,-2.597333,8.812000,5.717,5.717000,1998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113066,2015-05-28,924:MT:SNTL,44.65866,-111.09199,0.0,4.934,4.412,16.138000,2035,1.305896,-0.141421,-0.989949,4.387333,15.328000,12.286,369.302002,2015
113067,2015-05-29,924:MT:SNTL,44.65866,-111.09199,0.0,4.307,1.766,11.901000,2035,1.305896,-0.141421,-0.989949,3.633000,13.822667,13.262,373.609009,2015
113068,2015-05-30,924:MT:SNTL,44.65866,-111.09199,0.0,0.038,-0.797,19.499001,2035,1.305896,-0.141421,-0.989949,1.793667,15.846000,9.279,373.647003,2015
113069,2015-05-31,924:MT:SNTL,44.65866,-111.09199,0.0,5.639,1.194,20.554001,2035,1.305896,-0.141421,-0.989949,0.721000,17.318000,9.984,379.286011,2015


## Step 3: Split data into predictor (x) and predicted (y) variables

In [12]:
# split training data
x_train = stations_train[[
    # lat and lon
    'latitude', 'longitude', 
    # raw prism data
    'prism_ppt_mm', 'prism_tmin_c', 'prism_tmax_c', 
    # topographic data
    'elevation', 'slope', 'aspect_north', 'aspect_east', 
    # lagged prism variables
    'prism_tmin_c_roll_3d', 'prism_tmax_c_roll_3d', 'prism_ppt_mm_roll_3d', 
    # cumulative precip
    'wy_cumul_precip_mm']]
y_train = stations_train[['swe_mm']]

# split testing data
x_test = stations_test[[
    # lat and lon
    'latitude', 'longitude', 
    # raw prism data
    'prism_ppt_mm', 'prism_tmin_c', 'prism_tmax_c', 
    # topographic data
    'elevation', 'slope', 'aspect_north', 'aspect_east', 
    # lagged prism variables
    'prism_tmin_c_roll_3d', 'prism_tmax_c_roll_3d', 'prism_ppt_mm_roll_3d', 
    # cumulative precip
    'wy_cumul_precip_mm']]
y_test = stations_test[['swe_mm']]

In [14]:
# check that SWE is in y_train and y_test, not x_train and x_test

dfs = {
    'x_train': x_train,
    'y_train': y_train,
    'x_test': x_test,
    'y_test': y_test
}

for name, df in dfs.items():
    if 'swe_mm' in df.columns:
        print(f"SWE found in {name}")
    else:
        print(f"SWE not found in {name}")

SWE not found in x_train
SWE found in y_train
SWE not found in x_test
SWE found in y_test


## Step 4: Train Random Forests Model

In [15]:
# set up Random Forests model
rf_model = RandomForestRegressor(
    # 
    n_estimators=300,
    #
    max_depth=12,
    #
    max_features='sqrt',
    # set random state for repeatability
    random_state=42
)

In [21]:
# fit model
rf_model.fit(x_train, y_train)

# predict
rf_preds = rf_model.predict(x_test)

c:\Users\raini\miniconda3\envs\earth-analytics-python\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [ ]:
# Calculate some error statistics
rf_rmse = root_mean_squared_error(y_test, rf_preds)
rf_r2 = r2_score(y_test, rf_preds)

# check error statistics
print(f"RF Test RMSE: {rf_rmse} mm")
print(f"RF Test R^2: {rf_r2} ")

RF Test RMSE: 121.7656146397554
RF Test R^2: 0.7100215756465089 
